# 02: Explore the data (part 1: tables, counts and first decisions)

**What this notebook does**
1. Copies both files into DuckDB tables, so we can query them quickly and change them later (e.g. drop columns).
2. Counts rows, and checks that the join key `parent_asin` is safe.
3. Looks at `helpful_vote` and `main_category` to decide what to keep.

**Before you run it:** run notebook 01 first in the same Colab session, so the files and the `review_file` / `meta_file` names exist.

**Decisions from this notebook** are listed, with reasons, in [docs/06-decisions.md](../docs/06-decisions.md).

## Step 1: Copy both files into DuckDB tables

`CREATE TABLE ... AS SELECT` reads the file and stores it as a table inside DuckDB.
A table, unlike the original file, can be changed later (for example with `ALTER TABLE ... DROP COLUMN`).

**Warning: where the tables live.** `duckdb.sql(...)` uses DuckDB's default **in-memory** database, so these tables sit in RAM and are lost if the session restarts. That worked here, but it is risky with 20 million reviews. A safer option for later: open a database file with `duckdb.connect("amazon.duckdb")`, so the tables live on disk. See [docs/04-colab-setup.md](../docs/04-colab-setup.md).

The reading step takes about a minute or more per file.

In [ ]:
# Copy the review file into a DuckDB table called "review".
# read_json reads the zipped JSON Lines file directly; SELECT * takes every column.
duckdb.sql(f"CREATE TABLE review AS SELECT * FROM read_json('{review_file}')")

In [ ]:
# Copy the meta (product) file into a DuckDB table called "meta", the same way.
duckdb.sql(f"CREATE TABLE meta AS SELECT * FROM read_json('{meta_file}')")

## Step 2: Preview a few rows as a proper table

DuckDB's own printout wraps long review text across the screen, which looks messy but is only the display. The data itself is stored correctly.
For a tidy view: take a **small** piece with `LIMIT`, turn it into pandas with `.df()`, and use Colab's interactive table.

**Always keep the `LIMIT`.** `.df()` on the whole table would pull millions of rows into pandas and crash Colab.

In [ ]:
# Turn on Colab's Excel-like table view (sort, filter, scroll).
from google.colab import data_table
data_table.enable_dataframe_formatter()

# Take only the first 5 products, as a pandas table, and show it.
meta_preview = duckdb.sql("SELECT * FROM meta LIMIT 5").df()
meta_preview

## Step 3: Count the rows

In [ ]:
# How many reviews? (One row = one review.)
duckdb.sql("SELECT COUNT(*) AS total_rows FROM review")

In [ ]:
# How many products? (One row = one product.)
duckdb.sql("SELECT COUNT(*) AS total_rows FROM meta")

**Result:** 19,955,450 reviews and 2,003,129 products.

## Step 4: Is the join key safe?

Many reviews point to one product, so repeated `parent_asin` values in the **review** table are expected.
What must be true is that each product appears **once** in the **meta** table. Otherwise the join would copy reviews.

Check: the number of **distinct** `parent_asin` values in meta should equal the number of rows.

In [ ]:
# Count how many DIFFERENT parent_asin values meta has.
# Duplicates = total rows minus this number.
duckdb.sql("SELECT COUNT(DISTINCT parent_asin) AS unique_parent_asin FROM meta")

**Result:** 2,003,129, the same as the row count. So:
- no product appears twice, and
- no product is missing its ID (`COUNT(DISTINCT ...)` skips blanks, so a blank would have made this number smaller).

## Step 5: helpful_vote

In [ ]:
# How many DIFFERENT vote numbers appear in helpful_vote?
duckdb.sql("SELECT COUNT(DISTINCT helpful_vote) AS unique_vote FROM review")

**Result:** 741 different values.

Note: this counts **variety**, not frequency. It doesn't show how many reviews have 0 votes.

**Decision: drop `helpful_vote`.** The reason is **data leakage**, not the 741: votes pile up after a review is posted, right up to the 2023 collection date. The model must predict from early information only, so this column would bring in information from the future.

## Step 6: main_category

In [ ]:
# How many DIFFERENT main categories are there? (COUNT DISTINCT skips NULL.)
duckdb.sql("SELECT COUNT(DISTINCT main_category) FROM meta")

In [ ]:
# How many products are in each main category, biggest first.
# .df() is safe here: the result is only 39 rows.
category_query = '''
    SELECT main_category,
           COUNT(*) AS product_count
    FROM meta
    GROUP BY main_category
    ORDER BY product_count DESC
'''
category_table = duckdb.sql(category_query).df()
category_table

**Result:** 38 categories plus NULL (no category). That's 39 groups; `COUNT(DISTINCT ...)` showed 38 because it skips NULL.

| main_category | products | share |
|---|---|---|
| Automotive | 1,717,837 | about 85.8% |
| Industrial & Scientific | 58,909 | |
| NULL | 34,116 | about 1.7% |
| AMAZON FASHION | 26,594 | |
| Sports & Outdoors | 19,649 | |
| (34 smaller groups) | | |

**Decision:** keep only products whose `main_category` is **Automotive**, and exclude NULL. Then drop the `main_category` column, since every row would say the same thing.
*Planned, not yet applied: see "Next" below.*

## What we found in this notebook

- `parent_asin` is unique and never blank in meta, so it's safe as the join key.
- About 86% of products are Automotive; the rest belong to other categories.
- From the previews: `store` uses `"Unknown"` as a fake blank, `price` is often missing, and review text contains web codes like `&#34;` (meaning `"`) that need converting before sentiment analysis.

## Next (part 2)

1. Count how many products have `subtitle` / `author` filled in, then drop them.
2. Apply the Automotive filter and the column drops listed in [docs/06-decisions.md](../docs/06-decisions.md).
3. Join checks: how many reviews find no product, and the row count after a left join.
4. Look at how many reviews products have, and how fast they arrive, to set the target's early window.